In [1]:
import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

/tmp/ipykernel_2609392/2419554594.py:5: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth.chat_templates import get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
accelerate                1.9.0
bitsandbytes              0.46.1
datasets                  3.6.0
ipython                   8.37.0
jupyter                   1.1.1
jupyter_client            8.6.3
jupyter-console           6.6.3
jupyter_core              5.8.1
jupyter-events            0.12.0
jupyter-lsp               2.2.6
jupyter_server            2.16.0
jupyter_server_terminals  0.5.3
jupyterlab                4.4.5
jupyterlab_pygments       0.3.0
jupyterlab_server         2.27.3
jupyterlab_widgets        3.0.15
matplotlib                3.10.3
matplotlib-inline         0.1.7
notebook                  7.4.4
notebook_shim             0.2.4
numpy                     2.1.2
pandas                    2.3.1
peft                      0.16.0
torch                     2.6.0+cu118
torchaudio                2.6.0+cu118
torchvision               0.21.0+cu118
tqd

In [2]:
# Load model
# max_seq_length = 2048
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
#     token="",
#     max_seq_length=max_seq_length,
#     load_in_4bit=True,
#     dtype=None,
# )
#
# # Prepare model for PEFT
# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16,
#     lora_alpha=16,
#     lora_dropout=0,
#     target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
#     use_rslora=True,
#     use_gradient_checkpointing="unsloth"
# )
# print(model.print_trainable_parameters())


In [3]:
# # Define save directory (change as needed)
# save_dir = "./llama3-8b-4bit-unsloth"
#
# # Save model and tokenizer
# model.save_pretrained(
#     "./llama3-8b-4bit-unsloth",
#     save_embedding_layers=True,  # Saves base model
#     save_adapter=True,           # Saves LoRA
# )
# tokenizer.save_pretrained(save_dir)
# print(f"Model saved to: {save_dir}")

In [4]:

save_dir = "./llama3-8b-4bit-unsloth"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = save_dir,
    max_seq_length = 2048,  # Same as when you saved it
    load_in_4bit = True    # Keep it 4-bit quantized
)


# Then load the LoRA adapter
model = FastLanguageModel.get_peft_model(
        model,
            r = 16,  # Same rank as before
            lora_alpha = 16,  # Same alpha as before
            target_modules = [
                "q_proj", "k_proj", "v_proj", 
                "up_proj", "down_proj", 
                "o_proj", "gate_proj"
            ],  # Same target modules
            lora_dropout = 0,  # Same dropout
            use_rslora = True,  # Same RSLoRA setting
            use_gradient_checkpointing = "unsloth",  # Same checkpointing
            adapter_name = "default",  # Default adapter name
)
# Load the saved adapter weights
model.load_adapter(save_dir, adapter_name="default")  # Specify adapter_name

# Verify loading
print("Adapter loaded:", model.active_adapter)  # Should print "default"

print(model.print_trainable_parameters())


==((====))==  Unsloth 2025.7.9: Fast Llama patching. Transformers: 4.54.0.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu118. CUDA: 6.0. CUDA Toolkit: 11.8. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.7.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


Adapter loaded: default
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


In [5]:

# Check basic info
# print(dataset)           # Prints dataset structure (number of rows, features)
#print(dataset[0])        # Inspect the first example
# print(dataset.features)  # See column names and data types


In [6]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}
)

def apply_template(examples):
    messages = examples["conversations"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

def format_chatml(example):
    system_prompt = "<|system|>\nYou are a helpful assistant.\n"
    dialogue = ""
    for message in example["conversations"]:
        role = message["from"]
        content = message["value"]
        if role == "human":
            dialogue += f"<|user|>\n{content}\n"
        elif role == "gpt":
            dialogue += f"<|assistant|>\n{content}\n"
    return {"text": system_prompt + dialogue}

dataset = load_dataset("mlabonne/FineTome-100k", split="train[:100]")
dataset = dataset.map(format_chatml, remove_columns=dataset.column_names)
#dataset = dataset.map(format_chatml, batched=True, num_proc=2, remove_columns=dataset.column_names)   
#dataset = dataset.map(apply_template, remove_columns=dataset.column_names)

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.


In [8]:
# print("Tokenizer class:", type(tokenizer).__name__)
# print("Tokenizer config:", tokenizer.init_kwargs)

model.tokenizer = tokenizer  # Explicitly attach the tokenizer

# print("Is tokenizer attached?", hasattr(model, "tokenizer"))  # Should return True
# print("Tokenizer class:", type(model.tokenizer).__name__)


In [9]:
# # Take one example from your dataset
# test_example = dataset[0]  
# print("Raw example:", test_example)
#
# # Tokenize the formatted text
# tokens = tokenizer.tokenize(formatted_text)
# ids = tokenizer.encode(formatted_text)
#
# print("\nTokenized:", tokens)
# print("Token IDs:", ids)
# print("Decoded back:", tokenizer.decode(ids))
#
# # Ensure EOS/BOS tokens appear where expected
# if tokenizer.eos_token not in formatted_text:
#     print(f"\nWarning: EOS token ({tokenizer.eos_token}) missing!")
# if tokenizer.bos_token not in formatted_text:
#     print(f"Warning: BOS token ({tokenizer.bos_token}) missing!")

In [10]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding=False,
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"])

In [ ]:

max_seq_length = 2048

tokenizer.model_max_length = max_seq_length

trainer=SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    formatting_func=None,  # Replaces dataset_text_field
    args=TrainingArguments( 
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        seed=0,
    ),
)

trainer.train()